# Phase 1 — Data Ingestion & Storage

Load a CSV, remove duplicate review text, and store the cleaned dataset in **one PostgreSQL table** (`master_data`).

```bash
pip install pandas sqlalchemy psycopg2-binary
```

## Setup — Imports and project path

In [14]:
from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import Engine
from sqlalchemy.exc import SQLAlchemyError

PROJECT_DIR = Path.cwd().resolve()
MASTER_TABLE_NAME = "master_data"
print(f"Project directory: {PROJECT_DIR}")

Project directory: D:\NCPL\Bootcamp\Projects\Project 5\new data\data 2


---
## Step 1 — Read CSV from local folder

Prompts for CSV file name and custom header (comma-separated column names).

**Example** for `data.csv` (6 columns): `target,ids,date,flag,user,text`

In [15]:
def parse_header_input(raw: str) -> List[str]:
    raw = raw.strip()
    if raw.startswith("[") and raw.endswith("]"):
        raw = raw[1:-1]
    return [p.strip().strip("'\"").strip() for p in raw.split(",") if p.strip()]


def load_csv_with_custom_header(project_dir: Path) -> pd.DataFrame:
    csv_name = input("Enter CSV file name (e.g. data.csv): ").strip()
    if not csv_name:
        raise ValueError("CSV file name cannot be empty.")

    csv_path = project_dir / csv_name
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\n"
            f"Available: {[p.name for p in project_dir.glob('*.csv')] or '(none)'}"
        )

    header_raw = input(
        "Enter custom header (comma-separated):\n"
        "Example: target,ids,date,flag,user,text\n> "
    )
    custom_header = parse_header_input(header_raw)
    if not custom_header:
        raise ValueError("Header list is empty.")

    try:
        df = pd.read_csv(csv_path, header=None, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, header=None, encoding="latin-1", low_memory=False)

    if len(custom_header) != df.shape[1]:
        raise ValueError(
            f"Header length ({len(custom_header)}) != CSV columns ({df.shape[1]})."
        )

    df.columns = custom_header
    print(f"Loaded {len(df):,} rows × {len(df.columns)} columns from '{csv_name}'")
    display(df.head())
    return df


try:
    df_raw = load_csv_with_custom_header(PROJECT_DIR)
except (FileNotFoundError, ValueError) as exc:
    print(f"ERROR: {exc}", file=sys.stderr)
    raise

Loaded 1,600,000 rows × 6 columns from 'data.csv'


,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


---
## Step 2 — Clean duplicate reviews

Auto-detects the review text column and removes rows with **identical text** (`keep='first'`).

In [16]:
REVIEW_TEXT_CANDIDATES = [
    "review_text", "review", "text", "comment", "body",
    "content", "description", "message", "tweet",
]
USER_COLUMN_CANDIDATES = [
    "user", "username", "author", "reviewer", "customer", "screen_name", "handle",
]


def _match_column(columns: List[str], candidates: List[str]) -> str:
    normalized = {c: re.sub(r"[^a-z0-9]", "", c.lower()) for c in columns}
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", candidate)
        for col, norm in normalized.items():
            if norm == key or key in norm or norm in key:
                return col
    return ""


def find_review_text_column(columns: List[str]) -> str:
    col = _match_column(columns, REVIEW_TEXT_CANDIDATES)
    if col:
        return col
    for c in columns:
        if "text" in c.lower() or "review" in c.lower():
            return c
    raise ValueError(f"No review text column found. Columns: {columns}")


def find_user_column(columns: List[str]) -> str:
    col = _match_column(columns, USER_COLUMN_CANDIDATES)
    if col:
        return col
    raise ValueError(f"No user column found. Columns: {columns}")


def remove_duplicate_reviews(df: pd.DataFrame, review_col: str) -> Tuple[pd.DataFrame, int]:
    before = len(df)
    df_clean = df.drop_duplicates(subset=[review_col], keep="first").reset_index(drop=True)
    return df_clean, before - len(df_clean)


review_text_col = find_review_text_column(list(df_raw.columns))
user_col = find_user_column(list(df_raw.columns))
print(f"Review text column: '{review_text_col}'")
print(f"User column:        '{user_col}'")

df_clean, duplicates_removed = remove_duplicate_reviews(df_raw, review_text_col)

print(f"Rows before deduplication: {len(df_raw):,}")
print(f"Rows after deduplication:  {len(df_clean):,}")
print(f"Duplicates removed:        {duplicates_removed:,}")

Review text column: 'text'
User column:        'user'
Rows before deduplication: 1,600,000
Rows after deduplication:  1,581,466
Duplicates removed:        18,534


### Optional — Duplicate text by user (exploration)

Compares **same user + same text** vs **same text across different users** on `df_raw`.

In [17]:
same_user_pairs = (
    df_raw.groupby([user_col, review_text_col], dropna=False)
    .size()
    .reset_index(name="post_count")
    .query("post_count > 1")
)
same_user_extra = int(same_user_pairs["post_count"].sum() - len(same_user_pairs))

text_stats = (
    df_raw.groupby(review_text_col, dropna=False)[user_col]
    .agg(posts="count", unique_users="nunique")
    .reset_index()
    .query("posts > 1")
)
cross_user_extra = int(
    df_raw.groupby(review_text_col).size().reset_index(name="n").query("n > 1")["n"].sum()
    - len(text_stats)
)

global_text_dup_extra = len(df_raw) - df_raw.drop_duplicates(subset=[review_text_col]).shape[0]

print(f"Same user + same text pairs:     {len(same_user_pairs):,}")
print(f"Extra rows (same user + text):   {same_user_extra:,}")
print(f"Texts shared by 2+ users:        {len(text_stats.query('unique_users > 1')):,}")
print(f"Extra rows (cross-user text):    {cross_user_extra:,}")
print(f"Total removed in Step 2:         {global_text_dup_extra:,}")
display(same_user_pairs.head(10))

Same user + same text pairs:     3,547
Extra rows (same user + text):   5,255
Texts shared by 2+ users:        5,213
Extra rows (cross-user text):    18,534
Total removed in Step 2:         18,534


,user,text,post_count
24,007peter,"@NaniWaialeale No, not Keith Olbermann, he ann...",2
42,007wisdom,&quot;All that we are is the result of what we...,2
43,007wisdom,'Time Is An Illusion and All Time Is Now'... &...,4
762,1059StarFM,Big congrats to our Weight Loss Challenge cont...,2
1404,15AMR,I'm broke cuz she got all she wanted! Going ho...,2
1996,19fischi75,@Impala_Guy SORRY missed again - the boss was ...,2
2010,19fischi75,@Impala_Guy Yeah this is really annoying - wi...,2
2437,1_2_ManyTweets,Welcome home a Vietnam Vet just walk on up put...,5
2542,1brandingo,had a good time @ my birthday party tonight. t...,2
3143,2012worldshift,&quot;All that we are is the result of what we...,2


---
## Step 3 — Connect to PostgreSQL

In [18]:
def prompt_db_credentials() -> Dict[str, str]:
    return {
        "host": input("PostgreSQL host (e.g. localhost): ").strip(),
        "port": input("PostgreSQL port (5432): ").strip() or "5432",
        "database": input("Database name: ").strip(),
        "username": input("Username: ").strip(),
        "password": input("Password: ").strip(),
    }


def create_pg_engine(creds: Dict[str, str]) -> Engine:
    missing = [k for k in ("host", "database", "username", "password") if not creds.get(k)]
    if missing:
        raise ValueError(f"Missing connection fields: {missing}")
    url = (
        f"postgresql+psycopg2://{creds['username']}:{creds['password']}"
        f"@{creds['host']}:{creds['port']}/{creds['database']}"
    )
    return create_engine(url, pool_pre_ping=True)


def test_connection(engine: Engine) -> None:
    with engine.connect() as conn:
        print(conn.execute(text("SELECT version()")).scalar())
    print("Connection successful.")


db_creds = prompt_db_credentials()
try:
    engine = create_pg_engine(db_creds)
    test_connection(engine)
except (SQLAlchemyError, ValueError) as exc:
    print(f"Database connection failed: {exc}", file=sys.stderr)
    raise

PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit
Connection successful.


---
## Step 4 — Prepare master dataset

Keeps all original CSV columns and adds `row_id` as primary key. No extra normalized tables.

In [19]:
df_master = df_clean.reset_index(drop=True).copy()
df_master.insert(0, "row_id", range(1, len(df_master) + 1))

print(f"PostgreSQL table: {MASTER_TABLE_NAME}")
print(f"Rows: {len(df_master):,}  |  Columns: {list(df_master.columns)}")
display(df_master.head(3))

PostgreSQL table: master_data
Rows: 1,581,466  |  Columns: ['row_id', 'target', 'ids', 'date', 'flag', 'user', 'text']


,row_id,target,ids,date,flag,user,text
0,1,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,2,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,3,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...


---
## Step 5 — Store master table in PostgreSQL

Creates **only** `master_data`. Set `DROP_EXISTING = False` to append instead of replace.

In [20]:
DROP_EXISTING = True

from sqlalchemy import BigInteger, Column, Integer, MetaData, String, Table, Text

metadata = MetaData()
PG_INT32_MAX = 2_147_483_647
TEXT_COLUMN_PATTERN = re.compile(r"text|review|comment|body|tweet|message", re.I)


def _needs_bigint(series: pd.Series) -> bool:
    clean = series.dropna()
    return not clean.empty and (clean.max() > PG_INT32_MAX or clean.min() < -PG_INT32_MAX)


def _sql_type(series: pd.Series, col_name: str):
    if pd.api.types.is_integer_dtype(series):
        return BigInteger if _needs_bigint(series) else Integer
    if pd.api.types.is_float_dtype(series):
        non_null = series.dropna()
        if len(non_null) and non_null.map(lambda v: float(v).is_integer()).all():
            return BigInteger if _needs_bigint(non_null) else Integer
        return String(64)
    if TEXT_COLUMN_PATTERN.search(col_name):
        return Text
    max_len = int(series.astype(str).str.len().max()) if len(series) else 0
    return Text if max_len > 255 else String(min(max(max_len, 32), 1024)) if max_len else Text


def prepare_for_sql(df: pd.DataFrame) -> pd.DataFrame:
    out = df.replace([float("inf"), float("-inf")], pd.NA).copy()
    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            non_null = out[col].dropna()
            if len(non_null) and non_null.map(lambda v: float(v).is_integer()).all():
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
    return out


def create_and_load_master(engine: Engine, df: pd.DataFrame, table_name: str) -> int:
    cols = [Column("row_id", _sql_type(df["row_id"], "row_id"), primary_key=True, autoincrement=False)]
    cols += [Column(c, _sql_type(df[c], c), nullable=True) for c in df.columns if c != "row_id"]
    table = Table(table_name, metadata, *cols)
    if DROP_EXISTING:
        metadata.drop_all(engine, tables=[table], checkfirst=True)
    metadata.create_all(engine, tables=[table])
    df.to_sql(table_name, engine, if_exists="append", index=False, method="multi", chunksize=5000)
    with engine.connect() as conn:
        return conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()


try:
    df_sql = prepare_for_sql(df_master)
    master_row_count = create_and_load_master(engine, df_sql, MASTER_TABLE_NAME)
    print(f"Loaded '{MASTER_TABLE_NAME}': {master_row_count:,} rows")
except SQLAlchemyError as exc:
    print(f"Insert failed: {exc}", file=sys.stderr)
    raise

Loaded 'master_data': 1,581,466 rows


---
## Step 6 — Final output

In [21]:
def print_table_schema(engine: Engine, table_name: str) -> None:
    inspector = inspect(engine)
    print(f"\n=== Table: {table_name} ===")
    if not inspector.has_table(table_name):
        print("  (table not found)")
        return
    for col in inspector.get_columns(table_name):
        nullable = "NULL" if col.get("nullable") else "NOT NULL"
        print(f"  {col['name']:20} {col['type']!s:20} {nullable}")


print("=" * 60)
print("PHASE 1 — PIPELINE SUMMARY")
print("=" * 60)
print(f"Duplicates removed:  {duplicates_removed:,}")
print(f"Master table:        {MASTER_TABLE_NAME}")
print(f"Rows inserted:       {master_row_count:,}")
print_table_schema(engine, MASTER_TABLE_NAME)
print("\nPhase 1 complete.")

PHASE 1 — PIPELINE SUMMARY
Duplicates removed:  18,534
Master table:        master_data
Rows inserted:       1,581,466

=== Table: master_data ===
  row_id               INTEGER              NOT NULL
  target               INTEGER              NULL
  ids                  BIGINT               NULL
  date                 VARCHAR(32)          NULL
  flag                 VARCHAR(32)          NULL
  user                 VARCHAR(32)          NULL
  text                 TEXT                 NULL

Phase 1 complete.
